# RLPrompt from scratch
Runtime → Change runtime type → T4 GPU. Upload `rlprompt-from-scratch.zip` to the Colab file panel, then run the cells in order.

In [ ]:
!unzip -q -o rlprompt-from-scratch.zip -d /content
%cd /content/rlprompt-from-scratch
!pip install -q -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Offline tests
Tiny random models, runs on CPU in under a minute. Confirms the pipeline is wired correctly.

In [ ]:
!python -m pytest tests -q

## 2. Few-shot classification, SST-2
A 2-token prompt with 1,000 steps is a quick sanity run. The paper setting is `--prompt_length 5` with the default 12,000 steps.

In [ ]:
!python scripts/train_classification.py --dataset sst2 --prompt_length 2 --max_steps 1000 --eval_every 100 --output_dir outputs/sst2_quick

In [ ]:
import json
print(json.dumps(json.load(open('outputs/sst2_quick/results.json')), indent=2))

## 3. Prompt transfer across task LMs

In [ ]:
!python scripts/eval_classification.py --dataset sst2 --prompt_files outputs/sst2_quick/best_prompt.json --task_models roberta-base roberta-large gpt2 --out outputs/sst2_transfer.csv

## 4. Style transfer, Yelp negative → positive

In [ ]:
!python scripts/train_style_transfer.py --direction 0_to_1 --task_model distilgpt2 --max_steps 1000 --eval_every 100 --output_dir outputs/yelp_quick

In [ ]:
print(json.dumps(json.load(open('outputs/yelp_quick/results.json')), indent=2))
!head -15 outputs/yelp_quick/test_outputs.tsv

## 5. Style transfer baselines

In [ ]:
!python scripts/eval_style_transfer.py --baseline null --task_model distilgpt2 --out_dir outputs/yelp_baselines
!python scripts/eval_style_transfer.py --baseline manual --task_model distilgpt2 --out_dir outputs/yelp_baselines

## 7. Web app
Paste API keys below or type them in the app. Open the public `gradio.live` link that appears.

In [ ]:
import os
# os.environ['OPENAI_API_KEY'] = '...'
# os.environ['GEMINI_API_KEY'] = '...'
# os.environ['ANTHROPIC_API_KEY'] = '...'
!python app.py --share

## 6. Download results

In [ ]:
!zip -qr results.zip outputs
from google.colab import files
files.download('results.zip')